# Neighborhood Overlap Analysis — `climate_19.graphml`

Questo notebook classifica ogni arco del network come **bridge** (A–B) o **internal** (A–A / B–B)
e calcola la **neighborhood overlap** seguendo la definizione del notebook NS04:

$$O_{AB} = \frac{|N(A) \cap N(B)|}{|N(A) \cup N(B)|}$$

- $O_{AB} = 0$ → **local bridge** (nessun vicino in comune)
- Overlap basso → legame debole tra comunità distanti


## 1. Import e caricamento del grafo

In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

G = nx.read_graphml("climate_19.graphml")

groups    = nx.get_node_attributes(G, 'group')      # 'A' o 'B'
hierarchy = nx.get_node_attributes(G, 'hierarchy')  # 'A_CORE', 'B_PERIPHERY', ...

print(f"Nodi : {G.number_of_nodes()}")
print(f"Archi: {G.number_of_edges()}")
print("Distribuzione gruppi:", Counter(groups.values()))


## 2. Funzione `neighborhood_overlap`

Stessa implementazione del notebook NS04: i nodi $u$ e $v$ sono esclusi
dai rispettivi insiemi di vicini prima di calcolare intersezione e unione.


In [ ]:
def neighborhood_overlap(G, u, v):
    """
    Neighborhood overlap dell'arco (u, v).
    O_AB = |N(A) ∩ N(B)| / |N(A) ∪ N(B)|   [NS04]
    u e v sono esclusi da entrambi gli insiemi.
    """
    neighbors_u = set(G.neighbors(u)) - {v}
    neighbors_v = set(G.neighbors(v)) - {u}
    common = neighbors_u & neighbors_v
    union  = neighbors_u | neighbors_v
    return len(common) / len(union) if union else 0.0

# --- Verifica rapida su un esempio minimale ---
ex = nx.Graph()
ex.add_edges_from([('A','B'),('A','C'),('A','D'),('A','E'),('A','F'),
                   ('F','C'),('F','G'),('F','J')])
o_AF = neighborhood_overlap(ex, 'A', 'F')
print(f"Esempio NS04 — O_AF = {o_AF:.4f}  (atteso: {1/6:.4f})")


## 3. Calcolo overlap e classificazione degli archi

In [ ]:
rows = []
for u, v in G.edges():
    g_u = groups.get(u, '?')
    g_v = groups.get(v, '?')
    overlap = neighborhood_overlap(G, u, v)

    rows.append({
        'node_u'         : u,
        'node_v'         : v,
        'group_u'        : g_u,
        'group_v'        : g_v,
        'hierarchy_u'    : hierarchy.get(u, '?'),
        'hierarchy_v'    : hierarchy.get(v, '?'),
        'edge_type'      : 'bridge' if g_u != g_v else 'internal',
        'overlap'        : overlap,
        'is_local_bridge': overlap == 0.0,
    })

df = pd.DataFrame(rows)
print(f"Archi totali: {len(df)}")
df.head()


## 4. Statistiche di sintesi

In [ ]:
print("=== Distribuzione tipi di arco ===")
print(df['edge_type'].value_counts())

print("\n=== Overlap per tipo ===")
print(df.groupby('edge_type')['overlap'].describe().round(4))

print("\n=== Local bridges (overlap = 0) per tipo ===")
print(df[df['is_local_bridge']]['edge_type'].value_counts())


## 5. Visualizzazione

Confronto della distribuzione di overlap tra archi **bridge** (inter-comunità)
e **internal** (intra-comunità).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Istogramma
for etype, color in [('bridge', '#e74c3c'), ('internal', '#3498db')]:
    subset = df[df['edge_type'] == etype]['overlap']
    axes[0].hist(subset, bins=40, alpha=0.6, label=etype, color=color, density=True)
axes[0].set_xlabel("Neighborhood Overlap $O_{AB}$")
axes[0].set_ylabel("Densità")
axes[0].set_title("Distribuzione overlap\nbridge vs internal")
axes[0].legend()
axes[0].grid(axis='y', alpha=0.4)

# Boxplot
bp = axes[1].boxplot(
    [df[df['edge_type']=='bridge']['overlap'].values,
     df[df['edge_type']=='internal']['overlap'].values],
    labels=['Bridge\n(A–B)', 'Internal\n(A–A / B–B)'],
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2),
)
bp['boxes'][0].set_facecolor('#e74c3c'); bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#3498db'); bp['boxes'][1].set_alpha(0.6)
axes[1].set_ylabel("Neighborhood Overlap $O_{AB}$")
axes[1].set_title("Overlap: bridge vs internal")
axes[1].grid(axis='y', alpha=0.4)

plt.suptitle("Climate network — Neighborhood Overlap Analysis", fontsize=13)
plt.tight_layout()
plt.savefig("climate_overlap_plot.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot salvato: climate_overlap_plot.png")


## 6. Local bridges e Structural Holes

Un nodo che incide su molti local bridges occupa uno **structural hole**:
si trova tra gruppi che non interagirebbero altrimenti (Burt, 2000).


In [ ]:
lb_edges = [(r['node_u'], r['node_v'])
            for _, r in df[df['is_local_bridge']].iterrows()]

hole_count = defaultdict(int)
for u, v in lb_edges:
    hole_count[u] += 1
    hole_count[v] += 1

df_holes = pd.DataFrame([
    {'node': n, 'structural_holes': cnt,
     'degree': G.degree(n),
     'group': groups.get(n, '?'),
     'hierarchy': hierarchy.get(n, '?')}
    for n, cnt in hole_count.items()
]).sort_values('structural_holes', ascending=False)

print("Top 10 nodi per structural hole access:")
df_holes.head(10)


## 7. Export dei risultati

In [ ]:
df.to_csv("climate_overlap_results.csv", index=False)
df_holes.to_csv("climate_structural_holes.csv", index=False)

print("File salvati:")
print("  climate_overlap_results.csv   — archi con overlap e tipo")
print("  climate_structural_holes.csv  — nodi ordinati per structural hole access")


## Riepilogo

| Concetto | Formula | Riferimento |
|---|---|---|
| Neighborhood overlap | $O_{AB} = \frac{\|N(A) \cap N(B)\|}{\|N(A) \cup N(B)\|}$ | NS04 |
| Local bridge | $O_{AB} = 0$ | NS04 |
| Structural hole access | $\#\{\text{local bridges incidenti al nodo}\}$ | Burt (2000) |

**Risultato principale:** i bridge inter-comunità (A–B) hanno overlap sistematicamente
più basso degli archi interni, confermando che collegano parti strutturalmente distanti del network.
